# Spotter Freight Rate Prediction - Baseline Model Experiments

## Objective
Establish reproducible, leakage-safe baseline regression models for predicting spot freight `posted_rate`, evaluate temporal stability across rolling future-month holdouts, analyze feature scaling, conduct feature-group ablation studies, tune regularization penalties, perform comprehensive residual diagnostics, benchmark nonlinear gradient-boosted trees, test target transformations ($\log(1+y)$), evaluate leakage-safe historical pricing features, and test rate-per-mile target formulations.

### Experimental Agenda
1. **Part 1 - Single Temporal Holdout**: Baseline model comparison (Unscaled Ridge vs Random Forest) evaluated on October 2025 holdout (`cutoff_date="2025-10-01"`).
2. **Part 2 - Multi-Fold Temporal Validation**: Unscaled Ridge benchmark across rolling future-month windows (September 2025 holdout vs October 2025 holdout).
3. **Part 3 - Experiment A: Standardized Ridge**: Evaluating the impact of `StandardScaler` on numerical features across both temporal folds.
4. **Part 4 - Experiment B: Feature-Group Ablation**: Systematic step-up ablation on October holdout (Core -> Core+Temporal -> Core+Temporal+Spatial -> Core+Temporal+Spatial+Market Signals).
5. **Part 5 - Experiment C: Controlled Ridge Regularization**: Systematic evaluation of L2 penalty strength on Core features across both temporal folds.
6. **Part 6 - Experiment D: Residual & Error Analysis**: Deep dive into the error distribution of the best linear baseline (`Ridge(alpha=100.0)` on Core features).
7. **Part 7 - Experiment E: Nonlinear Gradient Boosting Benchmark**: Evaluating `HistGradientBoostingRegressor` with feature engineering, negative-weight handling, and hyperparameter tuning against the Ridge benchmark.
8. **Part 8 - Experiment F: Controlled Target-Transformation Benchmark**: Systematic evaluation of $\log(1 + 	ext{posted\_rate})$ target transformation with inverse $	ext{expm1}$ on Ridge and HistGradientBoosting.
9. **Part 9 - Experiment G: Leakage-Safe Historical Pricing Features**: Expanding historical lane pricing features (`route_prev_count`, `route_hist_median_rate`, etc.).
10. **Part 10 - Experiment H: Rate-Per-Mile Target Formulation**: Modeling $y_{\text{rpm}} = \frac{\text{posted\_rate}}{\text{distance}}$ and reconstructing $\hat{y} = \hat{y}_{\text{rpm}} \times \text{distance}$.

### Constraints Enforced
- Zero target leakage (no usage of current or future `posted_rate` or contemporaneous `rate_per_mile` in input features).
- Strict identifier exclusion (`load_id` dropped from feature set).
- Non-destructive processing (missing values handled via pipeline imputation, no row deletion).
- Evaluation Metrics: Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE).


In [1]:
import time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, median_absolute_error, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

# Import custom reusable project modules
from src.data import load_train_test
from src.features import (
    DERIVED_FEATURE_COLUMNS,
    HISTORICAL_FEATURE_COLUMNS,
    calculate_rate_per_mile_target,
    compute_historical_pricing_features,
    engineer_features,
    reconstruct_rate_from_rpm,
)
from src.validation import get_split_summary, temporal_train_valid_split

print("Modules imported successfully.")


Modules imported successfully.


## 1. Load Dataset
Load the primary `train-test.csv` dataset using `src.data.load_train_test`.


In [2]:
raw_df = load_train_test()
print(f"Raw dataset shape: {raw_df.shape}")
raw_df.head()


Raw dataset shape: (48000, 14)


,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,0.98480,2.56300,1380.28


## 2. Leakage-Safe Feature Engineering
Apply `src.features.engineer_features` to compute row-level temporal decompositions, route identifiers, and spatial coordinate features. Non-feature identifier `load_id` is excluded.


In [3]:
featured_df = engineer_features(raw_df, drop_load_id=True)
print(f"Featured dataset shape: {featured_df.shape}")
print(f"Engineered feature columns added: {list(DERIVED_FEATURE_COLUMNS)}")
featured_df.head()


Featured dataset shape: (48000, 25)
Engineered feature columns added: ['route', 'abs_lat_diff', 'abs_lon_diff', 'midpoint_lat', 'midpoint_lon', 'year', 'month', 'day', 'day_of_week', 'day_of_year', 'week_of_year', 'days_since_start']


,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,...,midpoint_lat,abs_lon_diff,midpoint_lon,year,month,day,day_of_week,day_of_year,week_of_year,days_since_start
0,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,...,38.130150,4.04342,-74.767350,2025,1,1,2,1,1,0
1,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,...,38.657195,3.82196,-74.878080,2025,1,1,2,1,1,0
2,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,...,41.763065,14.56161,-80.247905,2025,1,1,2,1,1,0
3,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,...,37.201305,14.10889,-79.234955,2025,1,1,2,1,1,0
4,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,...,33.562520,6.29428,-91.236290,2025,1,1,2,1,1,0


## 3. Initial Baseline: Unscaled Models on October 2025 Holdout
Partition data into training (Jan-Sep 2025) and validation (Oct 2025) to benchmark **Unscaled Ridge Regression** vs **Random Forest Regressor**.


In [4]:
X_train_oct, y_train_oct, X_valid_oct, y_valid_oct = temporal_train_valid_split(
    featured_df,
    cutoff_date="2025-10-01",
    target_column="posted_rate",
)

train_dates_oct = pd.to_datetime(X_train_oct["date"])
valid_dates_oct = pd.to_datetime(X_valid_oct["date"])

print(f"Training set:   {len(X_train_oct):,d} rows | Date range: {train_dates_oct.min().strftime('%Y-%m-%d')} to {train_dates_oct.max().strftime('%Y-%m-%d')}")
print(f"Validation set:  {len(X_valid_oct):,d} rows | Date range: {valid_dates_oct.min().strftime('%Y-%m-%d')} to {valid_dates_oct.max().strftime('%Y-%m-%d')}")


Training set:   43,147 rows | Date range: 2025-01-01 to 2025-09-30
Validation set:  4,853 rows | Date range: 2025-10-01 to 2025-10-31


### Preprocessing Pipeline Configuration (Unscaled)
- **Numerical Features**: Median imputation for missing values (`weight`, `market_index`, coordinate differences).
- **Categorical Features**: `OneHotEncoder(handle_unknown="ignore")` for categorical variables (`pickup`, `delivery`, `equipment`).


In [5]:
numeric_features = [
    "pickup_lat",
    "pickup_lon",
    "delivery_lat",
    "delivery_lon",
    "distance",
    "weight",
    "market_index",
    "quote_signal",
    "abs_lat_diff",
    "midpoint_lat",
    "abs_lon_diff",
    "midpoint_lon",
    "year",
    "month",
    "day",
    "day_of_week",
    "day_of_year",
    "week_of_year",
    "days_since_start",
]

categorical_features = ["pickup", "delivery", "equipment"]

unscaled_preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

print(f"Numerical feature count:   {len(numeric_features)}")
print(f"Categorical feature count: {len(categorical_features)}")


Numerical feature count:   19
Categorical feature count: 3


### Baseline 1: Unscaled Ridge Regression


In [6]:
ridge_pipeline = Pipeline([
    ("preprocessor", unscaled_preprocessor),
    ("regressor", Ridge(random_state=42)),
])

t0 = time.time()
ridge_pipeline.fit(X_train_oct, y_train_oct)
ridge_train_time = time.time() - t0

ridge_preds = ridge_pipeline.predict(X_valid_oct)
ridge_mae = mean_absolute_error(y_valid_oct, ridge_preds)
ridge_rmse = root_mean_squared_error(y_valid_oct, ridge_preds)

print(f"Unscaled Ridge Results (October 2025):")
print(f"  Training Time:   {ridge_train_time:.2f} s")
print(f"  Validation MAE:  ${ridge_mae:.2f}")
print(f"  Validation RMSE: ${ridge_rmse:.2f}")


Unscaled Ridge Results (October 2025):
  Training Time:   0.55 s
  Validation MAE:  $174.16
  Validation RMSE: $656.59


### Baseline 2: Random Forest Regressor


In [7]:
rf_pipeline = Pipeline([
    ("preprocessor", unscaled_preprocessor),
    ("regressor", RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)),
])

t0 = time.time()
rf_pipeline.fit(X_train_oct, y_train_oct)
rf_train_time = time.time() - t0

rf_preds = rf_pipeline.predict(X_valid_oct)
rf_mae = mean_absolute_error(y_valid_oct, rf_preds)
rf_rmse = root_mean_squared_error(y_valid_oct, rf_preds)

print(f"Random Forest Regressor Results (October 2025):")
print(f"  Training Time:   {rf_train_time:.2f} s")
print(f"  Validation MAE:  ${rf_mae:.2f}")
print(f"  Validation RMSE: ${rf_rmse:.2f}")


Random Forest Regressor Results (October 2025):
  Training Time:   400.11 s
  Validation MAE:  $242.72
  Validation RMSE: $760.27


### Single Holdout Model Comparison Table


In [8]:
transformed_feature_count = len(numeric_features) + ridge_pipeline.named_steps["preprocessor"].named_transformers_["cat"].get_feature_names_out().shape[0]

exp1_results = pd.DataFrame([
    {
        "Model": "Unscaled Ridge Regression (L2)",
        "Train Rows": len(X_train_oct),
        "Valid Rows": len(X_valid_oct),
        "Raw Features": len(numeric_features) + len(categorical_features),
        "Transformed Features": transformed_feature_count,
        "Validation MAE ($)": round(ridge_mae, 2),
        "Validation RMSE ($)": round(ridge_rmse, 2),
        "Training Time (s)": round(ridge_train_time, 2),
    },
    {
        "Model": "Random Forest Regressor",
        "Train Rows": len(X_train_oct),
        "Valid Rows": len(X_valid_oct),
        "Raw Features": len(numeric_features) + len(categorical_features),
        "Transformed Features": transformed_feature_count,
        "Validation MAE ($)": round(rf_mae, 2),
        "Validation RMSE ($)": round(rf_rmse, 2),
        "Training Time (s)": round(rf_train_time, 2),
    },
])

display(exp1_results)


,Model,Train Rows,Valid Rows,Raw Features,Transformed Features,Validation MAE ($),Validation RMSE ($),Training Time (s)
0,Unscaled Ridge Regression (L2),43147,4853,22,150,174.16,656.59,0.55
1,Random Forest Regressor,43147,4853,22,150,242.72,760.27,400.11


## 4. Multi-Fold Temporal Validation (Unscaled Ridge)
Evaluate Unscaled Ridge across rolling future-month windows:
- **FOLD 1**: Jan-Aug 2025 Train -> September 2025 Validation
- **FOLD 2**: Jan-Sep 2025 Train -> October 2025 Validation


In [9]:
# Fold 1: Train Jan-Aug 2025, Validate Sep 2025
df_fold1 = featured_df[pd.to_datetime(featured_df["date"]) < pd.Timestamp("2025-10-01")].copy()
X_tr1, y_tr1, X_val1, y_val1 = temporal_train_valid_split(
    df_fold1,
    cutoff_date="2025-09-01",
    target_column="posted_rate",
)

dates_tr1 = pd.to_datetime(X_tr1["date"])
dates_val1 = pd.to_datetime(X_val1["date"])

ridge_f1 = Pipeline([("preprocessor", unscaled_preprocessor), ("regressor", Ridge(random_state=42))])
ridge_f1.fit(X_tr1, y_tr1)
preds_f1 = ridge_f1.predict(X_val1)
mae_f1 = mean_absolute_error(y_val1, preds_f1)
rmse_f1 = root_mean_squared_error(y_val1, preds_f1)

# Fold 2: Train Jan-Sep 2025, Validate Oct 2025
X_tr2, y_tr2, X_val2, y_val2 = temporal_train_valid_split(
    featured_df,
    cutoff_date="2025-10-01",
    target_column="posted_rate",
)

dates_tr2 = pd.to_datetime(X_tr2["date"])
dates_val2 = pd.to_datetime(X_val2["date"])

ridge_f2 = Pipeline([("preprocessor", unscaled_preprocessor), ("regressor", Ridge(random_state=42))])
ridge_f2.fit(X_tr2, y_tr2)
preds_f2 = ridge_f2.predict(X_val2)
mae_f2 = mean_absolute_error(y_val2, preds_f2)
rmse_f2 = root_mean_squared_error(y_val2, preds_f2)

multifold_results = pd.DataFrame([
    {
        "Fold": "Fold 1",
        "Train End": dates_tr1.max().strftime('%Y-%m-%d'),
        "Validation Month": "September 2025",
        "Train Rows": len(X_tr1),
        "Valid Rows": len(X_val1),
        "MAE": round(mae_f1, 2),
        "RMSE": round(rmse_f1, 2),
    },
    {
        "Fold": "Fold 2",
        "Train End": dates_tr2.max().strftime('%Y-%m-%d'),
        "Validation Month": "October 2025",
        "Train Rows": len(X_tr2),
        "Valid Rows": len(X_val2),
        "MAE": round(mae_f2, 2),
        "RMSE": round(rmse_f2, 2),
    },
])

display(multifold_results)


,Fold,Train End,Validation Month,Train Rows,Valid Rows,MAE,RMSE
0,Fold 1,2025-08-31,September 2025,38477,4670,190.84,629.53
1,Fold 2,2025-09-30,October 2025,43147,4853,174.16,656.59


## 5. EXPERIMENT A: Standardized Ridge

### Objective
Evaluate the effect of feature scaling (`StandardScaler`) on numerical features within the preprocessing pipeline for Ridge regression across both temporal folds.


In [10]:
std_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_features,
        ),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

# Standardized Ridge - Fold 1 (September 2025)
std_ridge_f1 = Pipeline([
    ("preprocessor", std_preprocessor),
    ("regressor", Ridge(alpha=1.0, random_state=42)),
])
std_ridge_f1.fit(X_tr1, y_tr1)
preds_std_f1 = std_ridge_f1.predict(X_val1)
mae_std_f1 = mean_absolute_error(y_val1, preds_std_f1)
rmse_std_f1 = root_mean_squared_error(y_val1, preds_std_f1)

# Standardized Ridge - Fold 2 (October 2025)
std_ridge_f2 = Pipeline([
    ("preprocessor", std_preprocessor),
    ("regressor", Ridge(alpha=1.0, random_state=42)),
])
std_ridge_f2.fit(X_tr2, y_tr2)
preds_std_f2 = std_ridge_f2.predict(X_val2)
mae_std_f2 = mean_absolute_error(y_val2, preds_std_f2)
rmse_std_f2 = root_mean_squared_error(y_val2, preds_std_f2)

exp_a_comparison = pd.DataFrame([
    {
        "Fold": "Fold 1 (Sep 2025)",
        "Unscaled Ridge MAE ($)": round(mae_f1, 2),
        "Standardized Ridge MAE ($)": round(mae_std_f1, 2),
        "MAE Delta ($)": round(mae_std_f1 - mae_f1, 2),
        "Unscaled Ridge RMSE ($)": round(rmse_f1, 2),
        "Standardized Ridge RMSE ($)": round(rmse_std_f1, 2),
        "RMSE Delta ($)": round(rmse_std_f1 - rmse_f1, 2),
    },
    {
        "Fold": "Fold 2 (Oct 2025)",
        "Unscaled Ridge MAE ($)": round(mae_f2, 2),
        "Standardized Ridge MAE ($)": round(mae_std_f2, 2),
        "MAE Delta ($)": round(mae_std_f2 - mae_f2, 2),
        "Unscaled Ridge RMSE ($)": round(rmse_f2, 2),
        "Standardized Ridge RMSE ($)": round(rmse_std_f2, 2),
        "RMSE Delta ($)": round(rmse_std_f2 - rmse_f2, 2),
    },
])

display(exp_a_comparison)


,Fold,Unscaled Ridge MAE ($),Standardized Ridge MAE ($),MAE Delta ($),Unscaled Ridge RMSE ($),Standardized Ridge RMSE ($),RMSE Delta ($)
0,Fold 1 (Sep 2025),190.84,142.46,-48.37,629.53,623.17,-6.36
1,Fold 2 (Oct 2025),174.16,144.15,-30.01,656.59,651.86,-4.72


## 6. EXPERIMENT B: Feature-Group Ablation

### Objective
Systematically evaluate the predictive contribution of each feature group on the October holdout using Standardized Ridge.


In [11]:
ablation_groups = {
    "1. Core": {
        "num": ["distance", "weight"],
        "cat": ["pickup", "delivery", "equipment"],
    },
    "2. Core + Temporal": {
        "num": [
            "distance",
            "weight",
            "year",
            "month",
            "day",
            "day_of_week",
            "day_of_year",
            "week_of_year",
            "days_since_start",
        ],
        "cat": ["pickup", "delivery", "equipment"],
    },
    "3. Core + Temporal + Spatial": {
        "num": [
            "distance",
            "weight",
            "year",
            "month",
            "day",
            "day_of_week",
            "day_of_year",
            "week_of_year",
            "days_since_start",
            "pickup_lat",
            "pickup_lon",
            "delivery_lat",
            "delivery_lon",
            "abs_lat_diff",
            "midpoint_lat",
            "abs_lon_diff",
            "midpoint_lon",
        ],
        "cat": ["pickup", "delivery", "equipment"],
    },
    "4. Core + Temporal + Spatial + Market Signals": {
        "num": [
            "distance",
            "weight",
            "year",
            "month",
            "day",
            "day_of_week",
            "day_of_year",
            "week_of_year",
            "days_since_start",
            "pickup_lat",
            "pickup_lon",
            "delivery_lat",
            "delivery_lon",
            "abs_lat_diff",
            "midpoint_lat",
            "abs_lon_diff",
            "midpoint_lon",
            "market_index",
            "quote_signal",
        ],
        "cat": ["pickup", "delivery", "equipment"],
    },
}

ablation_rows = []

for name, grp in ablation_groups.items():
    num_cols = grp["num"]
    cat_cols = grp["cat"]
    raw_count = len(num_cols) + len(cat_cols)

    grp_preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]),
                num_cols,
            ),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ]
    )

    abl_pipe = Pipeline([
        ("preprocessor", grp_preprocessor),
        ("regressor", Ridge(alpha=1.0, random_state=42)),
    ])

    abl_pipe.fit(X_tr2, y_tr2)
    preds_abl = abl_pipe.predict(X_val2)
    mae_abl = mean_absolute_error(y_val2, preds_abl)
    rmse_abl = root_mean_squared_error(y_val2, preds_abl)

    cat_encoder = abl_pipe.named_steps["preprocessor"].named_transformers_["cat"]
    transformed_count = len(num_cols) + cat_encoder.get_feature_names_out().shape[0]

    ablation_rows.append({
        "Feature Group": name,
        "Raw Features": raw_count,
        "Transformed Features": transformed_count,
        "MAE ($)": round(mae_abl, 2),
        "RMSE ($)": round(rmse_abl, 2),
    })

ablation_df = pd.DataFrame(ablation_rows)
display(ablation_df)


,Feature Group,Raw Features,Transformed Features,MAE ($),RMSE ($)
0,1. Core,5,133,143.67,653.40
1,2. Core + Temporal,12,140,161.99,654.07
2,3. Core + Temporal + Spatial,20,148,161.58,654.02
3,4. Core + Temporal + Spatial + Market Signals,22,150,144.15,651.86


## 7. EXPERIMENT C: Controlled Ridge Regularization (Alpha Tuning)

### Objective
Systematically tune the L2 penalty parameter `alpha` for Ridge regression using **only the Core feature set** (`distance`, `weight`, `pickup`, `delivery`, `equipment`) evaluated across both temporal validation folds.


In [12]:
core_num_features = ["distance", "weight"]
core_cat_features = ["pickup", "delivery", "equipment"]
core_feature_cols = core_num_features + core_cat_features + ["date"]

core_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            core_num_features,
        ),
        ("cat", OneHotEncoder(handle_unknown="ignore"), core_cat_features),
    ]
)

# Extract core splits
df_core_f1 = raw_df[pd.to_datetime(raw_df["date"]) < pd.Timestamp("2025-10-01")].copy()
X_core_tr1, y_core_tr1, X_core_val1, y_core_val1 = temporal_train_valid_split(
    df_core_f1,
    cutoff_date="2025-09-01",
    target_column="posted_rate",
    feature_columns=core_feature_cols,
)

X_core_tr2, y_core_tr2, X_core_val2, y_core_val2 = temporal_train_valid_split(
    raw_df,
    cutoff_date="2025-10-01",
    target_column="posted_rate",
    feature_columns=core_feature_cols,
)

alphas = [0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0]
alpha_results = []

for alpha in alphas:
    # Fold 1
    pipe_f1 = Pipeline([
        ("preprocessor", core_preprocessor),
        ("regressor", Ridge(alpha=alpha, random_state=42)),
    ])
    pipe_f1.fit(X_core_tr1, y_core_tr1)
    preds_f1 = pipe_f1.predict(X_core_val1)
    mae_f1 = mean_absolute_error(y_core_val1, preds_f1)
    rmse_f1 = root_mean_squared_error(y_core_val1, preds_f1)

    # Fold 2
    pipe_f2 = Pipeline([
        ("preprocessor", core_preprocessor),
        ("regressor", Ridge(alpha=alpha, random_state=42)),
    ])
    pipe_f2.fit(X_core_tr2, y_core_tr2)
    preds_f2 = pipe_f2.predict(X_core_val2)
    mae_f2 = mean_absolute_error(y_core_val2, preds_f2)
    rmse_f2 = root_mean_squared_error(y_core_val2, preds_f2)

    mean_mae = (mae_f1 + mae_f2) / 2.0
    mean_rmse = (rmse_f1 + rmse_f2) / 2.0

    alpha_results.append({
        "alpha": alpha,
        "September MAE ($)": round(mae_f1, 2),
        "September RMSE ($)": round(rmse_f1, 2),
        "October MAE ($)": round(mae_f2, 2),
        "October RMSE ($)": round(rmse_f2, 2),
        "Mean MAE ($)": round(mean_mae, 2),
        "Mean RMSE ($)": round(mean_rmse, 2),
    })

alpha_df = pd.DataFrame(alpha_results).sort_values(by="Mean MAE ($)").reset_index(drop=True)
display(alpha_df)


,alpha,September MAE ($),September RMSE ($),October MAE ($),October RMSE ($),Mean MAE ($),Mean RMSE ($)
0,100.00,140.52,622.71,142.06,653.47,141.29,638.09
1,30.00,141.55,622.76,143.04,653.40,142.29,638.08
2,10.00,142.00,622.81,143.45,653.39,142.73,638.10
3,3.00,142.18,622.83,143.62,653.39,142.90,638.11
4,1.00,142.21,622.86,143.67,653.40,142.94,638.13
5,0.30,142.23,622.86,143.67,653.43,142.95,638.14
6,0.01,142.24,622.86,143.68,653.43,142.96,638.14
7,0.10,142.24,622.86,143.68,653.43,142.96,638.14
8,0.03,142.24,622.86,143.68,653.43,142.96,638.14


## 8. EXPERIMENT D: Residual & Error Analysis (Current Best Model: Ridge alpha=100)

### Objective
Perform detailed diagnostic error analysis on the best-performing linear baseline (`Ridge(alpha=100.0)` on Core features with standardized numerical inputs).


In [13]:
# Train best model pipeline on both folds
best_pipe_f1 = Pipeline([
    ("preprocessor", core_preprocessor),
    ("regressor", Ridge(alpha=100.0, random_state=42)),
])
best_pipe_f1.fit(X_core_tr1, y_core_tr1)
best_preds_f1 = best_pipe_f1.predict(X_core_val1)

best_pipe_f2 = Pipeline([
    ("preprocessor", core_preprocessor),
    ("regressor", Ridge(alpha=100.0, random_state=42)),
])
best_pipe_f2.fit(X_core_tr2, y_core_tr2)
best_preds_f2 = best_pipe_f2.predict(X_core_val2)

val_diag_f1 = df_core_f1[pd.to_datetime(df_core_f1["date"]) >= pd.Timestamp("2025-09-01")].copy().reset_index(drop=True)
val_diag_f1["predicted_rate"] = best_preds_f1
val_diag_f1["residual"] = best_preds_f1 - y_core_val1.values
val_diag_f1["abs_error"] = np.abs(val_diag_f1["residual"])

val_diag_f2 = raw_df[pd.to_datetime(raw_df["date"]) >= pd.Timestamp("2025-10-01")].copy().reset_index(drop=True)
val_diag_f2["predicted_rate"] = best_preds_f2
val_diag_f2["residual"] = best_preds_f2 - y_core_val2.values
val_diag_f2["abs_error"] = np.abs(val_diag_f2["residual"])

def calc_dist_metrics(y_true, y_pred, abs_err):
    return {
        "MAE ($)": mean_absolute_error(y_true, y_pred),
        "RMSE ($)": root_mean_squared_error(y_true, y_pred),
        "Median AE ($)": median_absolute_error(y_true, y_pred),
        "P90 AE ($)": np.percentile(abs_err, 90),
        "P95 AE ($)": np.percentile(abs_err, 95),
        "P99 AE ($)": np.percentile(abs_err, 99),
        "Max AE ($)": np.max(abs_err),
    }

diag_metrics_df = pd.DataFrame([
    {"Fold": "Fold 1 (Sep 2025)", **{k: round(v, 2) for k, v in calc_dist_metrics(y_core_val1, best_preds_f1, val_diag_f1["abs_error"]).items()}},
    {"Fold": "Fold 2 (Oct 2025)", **{k: round(v, 2) for k, v in calc_dist_metrics(y_core_val2, best_preds_f2, val_diag_f2["abs_error"]).items()}},
])

print("=== FOLD ERROR DISTRIBUTION METRICS ===")
display(diag_metrics_df)


=== FOLD ERROR DISTRIBUTION METRICS ===


,Fold,MAE ($),RMSE ($),Median AE ($),P90 AE ($),P95 AE ($),P99 AE ($),Max AE ($)
0,Fold 1 (Sep 2025),140.52,622.71,60.38,224.41,313.58,1338.41,16362.54
1,Fold 2 (Oct 2025),142.06,653.47,60.77,201.91,282.61,1813.70,13884.22


### Top 20 Worst-Error Cases (October Holdout)


In [14]:
worst20_oct = val_diag_f2.sort_values(by="abs_error", ascending=False).head(20)
display(worst20_oct[[
    "load_id", "pickup", "delivery", "equipment", "distance", "weight", "date",
    "market_index", "posted_rate", "predicted_rate", "residual", "abs_error"
]])


,load_id,pickup,delivery,equipment,distance,weight,date,market_index,posted_rate,predicted_rate,residual,abs_error
392,TR-043540,Milwaukee,Bakersfield,Dry Van,1893.0,30760.0,2025-10-03,1.08079,17514.11,3629.891543,-13884.218457,13884.218457
3066,TR-046214,Columbia,Tucson,Flatbed,2023.4,34710.0,2025-10-20,0.89822,17893.17,4101.303800,-13791.866200,13791.866200
4151,TR-047299,Boston,Bakersfield,Reefer,2979.1,30585.0,2025-10-27,0.86501,19110.30,5942.147088,-13168.152912,13168.152912
3369,TR-046517,Toledo,Albuquerque,Reefer,1684.3,31536.0,2025-10-22,0.98632,14784.38,3543.880934,-11240.499066,11240.499066
3219,TR-046367,Bakersfield,Columbia,Dry Van,2251.9,40729.0,2025-10-21,0.94743,14403.14,4358.300928,-10044.839072,10044.839072
4389,TR-047537,Los Angeles,Richmond,Dry Van,2782.4,25730.0,2025-10-29,1.03410,15003.55,5225.829409,-9777.720591,9777.720591
1120,TR-044268,Albany,Albuquerque,Dry Van,2492.6,25513.0,2025-10-08,1.05590,14425.42,4718.432394,-9706.987606,9706.987606
155,TR-043303,Atlanta,Phoenix,Reefer,2083.1,40335.0,2025-10-01,0.90740,13894.55,4367.298593,-9527.251407,9527.251407
237,TR-043385,Lexington,Lubbock,Dry Van,1187.0,25359.0,2025-10-02,0.95849,11696.44,2417.954507,-9278.485493,9278.485493
4439,TR-047587,Corpus Christi,Las Vegas,Flatbed,1383.3,18645.0,2025-10-29,1.01713,11596.28,2886.824590,-8709.455410,8709.455410


### Error Subgroup Segmentations (October Holdout)


In [15]:
# 1. Weight Category Segmentation
val_diag_f2["weight_category"] = np.where(
    val_diag_f2["weight"].isna(), "Missing",
    np.where(val_diag_f2["weight"] < 0, "Negative (<0)", "Normal (>=0)")
)
w_summary = val_diag_f2.groupby("weight_category")["abs_error"].agg(
    Count="count", Mean_AE="mean", Median_AE="median", RMSE=lambda x: np.sqrt(np.mean(x**2))
).reset_index()

# 2. Market Index Availability
val_diag_f2["market_index_status"] = np.where(val_diag_f2["market_index"].isna(), "Missing", "Present")
mi_summary = val_diag_f2.groupby("market_index_status")["abs_error"].agg(
    Count="count", Mean_AE="mean", Median_AE="median", RMSE=lambda x: np.sqrt(np.mean(x**2))
).reset_index()

# 3. Equipment Type
eq_summary = val_diag_f2.groupby("equipment")["abs_error"].agg(
    Count="count", Mean_AE="mean", Median_AE="median", RMSE=lambda x: np.sqrt(np.mean(x**2))
).reset_index()

# 4. Route Frequency in Training
train_route_counts = X_core_tr2["pickup"].str.cat(X_core_tr2["delivery"], sep="__").value_counts()
val_diag_f2["route"] = val_diag_f2["pickup"] + "__" + val_diag_f2["delivery"]
val_diag_f2["train_route_freq"] = val_diag_f2["route"].map(train_route_counts).fillna(0)
val_diag_f2["route_frequency_group"] = np.where(
    val_diag_f2["train_route_freq"] == 0, "Unseen Route (0)",
    np.where(val_diag_f2["train_route_freq"] < 10, "Rare Route (1-9)", "Common Route (>=10)")
)
rf_summary = val_diag_f2.groupby("route_frequency_group")["abs_error"].agg(
    Count="count", Mean_AE="mean", Median_AE="median", RMSE=lambda x: np.sqrt(np.mean(x**2))
).reset_index()

# 5. City Seen vs Unseen in Training
train_pickups = set(X_core_tr2["pickup"])
train_deliveries = set(X_core_tr2["delivery"])
val_diag_f2["city_status"] = np.where(
    val_diag_f2["pickup"].isin(train_pickups) & val_diag_f2["delivery"].isin(train_deliveries),
    "Both Cities Seen",
    "Unseen City"
)
city_summary = val_diag_f2.groupby("city_status")["abs_error"].agg(
    Count="count", Mean_AE="mean", Median_AE="median", RMSE=lambda x: np.sqrt(np.mean(x**2))
).reset_index()

# 6. Distance Quintiles
val_diag_f2["distance_bin"] = pd.qcut(val_diag_f2["distance"], q=5, labels=["Short (<450mi)", "Medium-Short", "Medium", "Medium-Long", "Long (>2000mi)"])
dist_summary = val_diag_f2.groupby("distance_bin", observed=False)["abs_error"].agg(
    Count="count", Mean_AE="mean", Median_AE="median", RMSE=lambda x: np.sqrt(np.mean(x**2))
).reset_index()

# 7. Posted Rate Quintiles
val_diag_f2["posted_rate_bin"] = pd.qcut(val_diag_f2["posted_rate"], q=5, labels=["Very Low Rate", "Low Rate", "Medium Rate", "High Rate", "Very High Rate"])
rate_summary = val_diag_f2.groupby("posted_rate_bin", observed=False)["abs_error"].agg(
    Count="count", Mean_AE="mean", Median_AE="median", RMSE=lambda x: np.sqrt(np.mean(x**2))
).reset_index()

print("--- 1. Weight Category Segmentation ---")
display(w_summary.round(2))

print("--- 2. Market Index Status ---")
display(mi_summary.round(2))

print("--- 3. Equipment Type ---")
display(eq_summary.round(2))

print("--- 4. Route Frequency in Training ---")
display(rf_summary.round(2))

print("--- 5. City Status ---")
display(city_summary.round(2))

print("--- 6. Distance Quintiles ---")
display(dist_summary.round(2))

print("--- 7. Posted Rate Quintiles ---")
display(rate_summary.round(2))


--- 1. Weight Category Segmentation ---


,weight_category,Count,Mean_AE,Median_AE,RMSE
0,Missing,32,94.38,73.57,121.75
1,Negative (<0),33,324.97,270.26,378.00
2,Normal (>=0),4788,141.12,60.38,657.07


--- 2. Market Index Status ---


,market_index_status,Count,Mean_AE,Median_AE,RMSE
0,Missing,42,134.79,64.20,386.74
1,Present,4811,142.12,60.75,655.32


--- 3. Equipment Type ---


...

,city_status,Count,Mean_AE,Median_AE,RMSE
0,Both Cities Seen,4853,142.06,60.77,653.47


--- 6. Distance Quintiles ---


,distance_bin,Count,Mean_AE,Median_AE,RMSE
0,Short (<450mi),972,117.22,71.26,272.68
1,Medium-Short,969,87.16,51.63,319.39
2,Medium,971,100.48,41.88,520.23
3,Medium-Long,970,151.80,58.78,766.35
4,Long (>2000mi),971,253.57,122.30,1049.13


--- 7. Posted Rate Quintiles ---


,posted_rate_bin,Count,Mean_AE,Median_AE,RMSE
0,Very Low Rate,971,140.94,68.67,351.81
1,Low Rate,970,66.65,50.79,178.12
2,Medium Rate,971,56.89,41.64,138.14
3,High Rate,970,76.70,60.07,147.52
4,Very High Rate,971,368.97,128.78,1392.10


## 9. EXPERIMENT E: Nonlinear Gradient Boosting Benchmark (`HistGradientBoostingRegressor`)

### Objective
Benchmark nonlinear gradient-boosted decision trees (`HistGradientBoostingRegressor`) against the current best Ridge model (`Ridge(alpha=100.0)`).


In [16]:
def prepare_hgb_features(df: pd.DataFrame, train_weight_median: float = None, train_market_median: float = None):
    out = df.copy()
    out["is_negative_weight"] = np.where(out["weight"] < 0, 1.0, 0.0)
    cleaned_weight = np.where(out["weight"] < 0, np.nan, out["weight"])
    if train_weight_median is None:
        train_weight_median = np.nanmedian(cleaned_weight)
    out["weight_cleaned"] = np.where(np.isnan(cleaned_weight), train_weight_median, cleaned_weight)
    if train_market_median is None:
        train_market_median = out["market_index"].median()
    market_imputed = out["market_index"].fillna(train_market_median)
    
    dist = out["distance"]
    out["log1p_distance"] = np.log1p(dist)
    out["distance_sq"] = (dist / 1000.0) ** 2
    out["weight_sq"] = (out["weight_cleaned"] / 10000.0) ** 2
    out["dist_x_market"] = dist * market_imputed
    out["dist_x_quote"] = dist * out["quote_signal"]
    eq_mult = {"Dry Van": 1.0, "Flatbed": 1.15, "Reefer": 1.35}
    out["dist_x_equipment"] = dist * out["equipment"].map(eq_mult).fillna(1.0)
    
    feature_cols = [
        "distance", "weight_cleaned", "is_negative_weight",
        "pickup_lat", "pickup_lon", "delivery_lat", "delivery_lon",
        "market_index", "quote_signal",
        "log1p_distance", "distance_sq", "weight_sq",
        "dist_x_market", "dist_x_quote", "dist_x_equipment",
        "pickup", "delivery", "equipment",
    ]
    return out[feature_cols], train_weight_median, train_market_median

# Prepare fold feature matrices
X_hgb_tr1, w_med1, m_med1 = prepare_hgb_features(X_tr1)
X_hgb_val1, _, _ = prepare_hgb_features(X_val1, train_weight_median=w_med1, train_market_median=m_med1)

X_hgb_tr2, w_med2, m_med2 = prepare_hgb_features(X_tr2)
X_hgb_val2, _, _ = prepare_hgb_features(X_val2, train_weight_median=w_med2, train_market_median=m_med2)

hgb_cat_cols = ["pickup", "delivery", "equipment"]
enc1 = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_hgb_tr1[hgb_cat_cols] = enc1.fit_transform(X_hgb_tr1[hgb_cat_cols])
X_hgb_val1[hgb_cat_cols] = enc1.transform(X_hgb_val1[hgb_cat_cols])

enc2 = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_hgb_tr2[hgb_cat_cols] = enc2.fit_transform(X_hgb_tr2[hgb_cat_cols])
X_hgb_val2[hgb_cat_cols] = enc2.transform(X_hgb_val2[hgb_cat_cols])

hgb_cat_indices = [X_hgb_tr1.columns.get_loc(c) for c in hgb_cat_cols]

# Hyperparameter search grid
hgb_param_grid = [
    {"learning_rate": 0.03, "max_iter": 200, "max_leaf_nodes": 15, "l2_regularization": 0.0},
    {"learning_rate": 0.03, "max_iter": 400, "max_leaf_nodes": 31, "l2_regularization": 1.0},
    {"learning_rate": 0.05, "max_iter": 200, "max_leaf_nodes": 15, "l2_regularization": 0.0},
    {"learning_rate": 0.05, "max_iter": 200, "max_leaf_nodes": 31, "l2_regularization": 1.0},
    {"learning_rate": 0.05, "max_iter": 400, "max_leaf_nodes": 31, "l2_regularization": 0.0},
    {"learning_rate": 0.05, "max_iter": 400, "max_leaf_nodes": 31, "l2_regularization": 1.0},
    {"learning_rate": 0.10, "max_iter": 200, "max_leaf_nodes": 15, "l2_regularization": 0.0},
    {"learning_rate": 0.10, "max_iter": 200, "max_leaf_nodes": 31, "l2_regularization": 1.0},
    {"learning_rate": 0.10, "max_iter": 400, "max_leaf_nodes": 31, "l2_regularization": 1.0},
]

hgb_results = []

for p in hgb_param_grid:
    m1 = HistGradientBoostingRegressor(
        learning_rate=p["learning_rate"],
        max_iter=p["max_iter"],
        max_leaf_nodes=p["max_leaf_nodes"],
        l2_regularization=p["l2_regularization"],
        categorical_features=hgb_cat_indices,
        random_state=42,
    )
    m1.fit(X_hgb_tr1, y_tr1)
    preds1 = m1.predict(X_hgb_val1)
    mae1 = mean_absolute_error(y_val1, preds1)
    rmse1 = root_mean_squared_error(y_val1, preds1)

    m2 = HistGradientBoostingRegressor(
        learning_rate=p["learning_rate"],
        max_iter=p["max_iter"],
        max_leaf_nodes=p["max_leaf_nodes"],
        l2_regularization=p["l2_regularization"],
        categorical_features=hgb_cat_indices,
        random_state=42,
    )
    m2.fit(X_hgb_tr2, y_tr2)
    preds2 = m2.predict(X_hgb_val2)
    mae2 = mean_absolute_error(y_val2, preds2)
    rmse2 = root_mean_squared_error(y_val2, preds2)

    mean_mae = (mae1 + mae2) / 2.0
    mean_rmse = (rmse1 + rmse2) / 2.0

    hgb_results.append({
        "learning_rate": p["learning_rate"],
        "max_iter": p["max_iter"],
        "max_leaf_nodes": p["max_leaf_nodes"],
        "l2_reg": p["l2_regularization"],
        "September MAE ($)": round(mae1, 2),
        "September RMSE ($)": round(rmse1, 2),
        "October MAE ($)": round(mae2, 2),
        "October RMSE ($)": round(rmse2, 2),
        "Mean MAE ($)": round(mean_mae, 2),
        "Mean RMSE ($)": round(mean_rmse, 2),
    })

hgb_res_df = pd.DataFrame(hgb_results).sort_values(by="Mean MAE ($)").reset_index(drop=True)
print("=== HIST GRADIENT BOOSTING HYPERPARAMETER GRID RESULTS ===")
display(hgb_res_df)


=== HIST GRADIENT BOOSTING HYPERPARAMETER GRID RESULTS ===


,learning_rate,max_iter,max_leaf_nodes,l2_reg,September MAE ($),September RMSE ($),October MAE ($),October RMSE ($),Mean MAE ($),Mean RMSE ($)
0,0.10,200,15,0.0,149.60,628.93,137.81,656.09,143.70,642.51
1,0.05,200,15,0.0,151.16,628.65,137.02,655.80,144.09,642.23
2,0.03,200,15,0.0,153.80,630.39,138.99,655.69,146.39,643.04
3,0.03,400,31,1.0,150.12,628.36,145.33,659.39,147.72,643.87
4,0.05,400,31,0.0,152.36,628.20,146.19,660.09,149.28,644.14
5,0.05,200,31,1.0,150.76,628.86,147.86,660.63,149.31,644.74
6,0.05,400,31,1.0,150.76,628.86,147.86,660.63,149.31,644.74
7,0.10,200,31,1.0,153.49,628.88,147.99,660.81,150.74,644.84
8,0.10,400,31,1.0,153.49,628.88,147.99,660.81,150.74,644.84


### Comparison: Best HistGradientBoosting vs. Ridge Benchmark


In [17]:
best_hgb_f1 = HistGradientBoostingRegressor(
    learning_rate=0.10, max_iter=200, max_leaf_nodes=15, l2_regularization=0.0,
    categorical_features=hgb_cat_indices, random_state=42
)
best_hgb_f1.fit(X_hgb_tr1, y_tr1)
best_hgb_preds_f1 = best_hgb_f1.predict(X_hgb_val1)

best_hgb_f2 = HistGradientBoostingRegressor(
    learning_rate=0.10, max_iter=200, max_leaf_nodes=15, l2_regularization=0.0,
    categorical_features=hgb_cat_indices, random_state=42
)
best_hgb_f2.fit(X_hgb_tr2, y_tr2)
best_hgb_preds_f2 = best_hgb_f2.predict(X_hgb_val2)

val_diag_f2["hgb_pred"] = best_hgb_preds_f2
val_diag_f2["hgb_abs_err"] = np.abs(best_hgb_preds_f2 - y_val2.values)
val_diag_f2["ridge_abs_err"] = np.abs(best_preds_f2 - y_val2.values)

m_ridge_oct = calc_dist_metrics(y_val2, best_preds_f2, val_diag_f2["ridge_abs_err"])
m_hgb_oct = calc_dist_metrics(y_val2, best_hgb_preds_f2, val_diag_f2["hgb_abs_err"])

hgb_vs_ridge = pd.DataFrame([
    {"Model": "Standardized Ridge (alpha=100)", **{k: round(v, 2) for k, v in m_ridge_oct.items()}},
    {"Model": "HistGradientBoosting (best)", **{k: round(v, 2) for k, v in m_hgb_oct.items()}},
])

print("=== OCTOBER HOLD-OUT RESIDUAL DIAGNOSTICS: RIDGE VS. BOOSTING ===")
display(hgb_vs_ridge)


=== OCTOBER HOLD-OUT RESIDUAL DIAGNOSTICS: RIDGE VS. BOOSTING ===


,Model,MAE ($),RMSE ($),Median AE ($),P90 AE ($),P95 AE ($),P99 AE ($),Max AE ($)
0,Standardized Ridge (alpha=100),142.06,653.47,60.77,201.91,282.61,1813.70,13884.22
1,HistGradientBoosting (best),137.81,656.09,52.89,188.29,278.50,1775.68,13828.73


## 10. EXPERIMENT F: Controlled Target-Transformation Benchmark ($\log(1 + y)$)

### Objective
Evaluate the impact of modeling in log-target space:
$$y_{\log} = \log(1 + \text{posted\_rate}) \quad \Longleftrightarrow \quad \hat{y} = \exp(\hat{y}_{\log}) - 1$$


In [18]:
target_exp_results = []

target_exp_results.append({
    "Model": "Linear Ridge (alpha=100) [Benchmark]",
    "September MAE ($)": round(mean_absolute_error(y_val1, best_preds_f1), 2),
    "September RMSE ($)": round(root_mean_squared_error(y_val1, best_preds_f1), 2),
    "October MAE ($)": round(mean_absolute_error(y_val2, best_preds_f2), 2),
    "October RMSE ($)": round(root_mean_squared_error(y_val2, best_preds_f2), 2),
    "Mean MAE ($)": round((mean_absolute_error(y_val1, best_preds_f1) + mean_absolute_error(y_val2, best_preds_f2)) / 2, 2),
    "Mean RMSE ($)": round((root_mean_squared_error(y_val1, best_preds_f1) + root_mean_squared_error(y_val2, best_preds_f2)) / 2, 2),
})

target_exp_results.append({
    "Model": "Linear HistGradientBoosting (best) [Benchmark]",
    "September MAE ($)": round(mean_absolute_error(y_val1, best_hgb_preds_f1), 2),
    "September RMSE ($)": round(root_mean_squared_error(y_val1, best_hgb_preds_f1), 2),
    "October MAE ($)": round(mean_absolute_error(y_val2, best_hgb_preds_f2), 2),
    "October RMSE ($)": round(root_mean_squared_error(y_val2, best_hgb_preds_f2), 2),
    "Mean MAE ($)": round((mean_absolute_error(y_val1, best_hgb_preds_f1) + mean_absolute_error(y_val2, best_hgb_preds_f2)) / 2, 2),
    "Mean RMSE ($)": round((root_mean_squared_error(y_val1, best_hgb_preds_f1) + root_mean_squared_error(y_val2, best_hgb_preds_f2)) / 2, 2),
})

# 2. Log-Target Ridge
for alpha_val in [10.0, 30.0, 100.0]:
    log_ridge_pipe1 = Pipeline([
        ("preprocessor", core_preprocessor),
        ("regressor", TransformedTargetRegressor(
            regressor=Ridge(alpha=alpha_val, random_state=42),
            func=np.log1p,
            inverse_func=np.expm1,
        )),
    ])
    log_ridge_pipe1.fit(X_core_tr1, y_core_tr1)
    p1 = log_ridge_pipe1.predict(X_core_val1)

    log_ridge_pipe2 = Pipeline([
        ("preprocessor", core_preprocessor),
        ("regressor", TransformedTargetRegressor(
            regressor=Ridge(alpha=alpha_val, random_state=42),
            func=np.log1p,
            inverse_func=np.expm1,
        )),
    ])
    log_ridge_pipe2.fit(X_core_tr2, y_core_tr2)
    p2 = log_ridge_pipe2.predict(X_core_val2)

    m_mae1 = mean_absolute_error(y_val1, p1)
    m_rmse1 = root_mean_squared_error(y_val1, p1)
    m_mae2 = mean_absolute_error(y_val2, p2)
    m_rmse2 = root_mean_squared_error(y_val2, p2)

    target_exp_results.append({
        "Model": f"Log-Target Ridge (alpha={int(alpha_val)})",
        "September MAE ($)": round(m_mae1, 2),
        "September RMSE ($)": round(m_rmse1, 2),
        "October MAE ($)": round(m_mae2, 2),
        "October RMSE ($)": round(m_rmse2, 2),
        "Mean MAE ($)": round((m_mae1 + m_mae2) / 2, 2),
        "Mean RMSE ($)": round((m_rmse1 + m_rmse2) / 2, 2),
    })

# 3. Log-Target HistGradientBoosting
for lr_val in [0.10, 0.05, 0.03]:
    log_hgb_model1 = TransformedTargetRegressor(
        regressor=HistGradientBoostingRegressor(
            learning_rate=lr_val,
            max_iter=200,
            max_leaf_nodes=15,
            l2_regularization=0.0,
            categorical_features=hgb_cat_indices,
            random_state=42,
        ),
        func=np.log1p,
        inverse_func=np.expm1,
    )
    log_hgb_model1.fit(X_hgb_tr1, y_tr1)
    p1 = log_hgb_model1.predict(X_hgb_val1)

    log_hgb_model2 = TransformedTargetRegressor(
        regressor=HistGradientBoostingRegressor(
            learning_rate=lr_val,
            max_iter=200,
            max_leaf_nodes=15,
            l2_regularization=0.0,
            categorical_features=hgb_cat_indices,
            random_state=42,
        ),
        func=np.log1p,
        inverse_func=np.expm1,
    )
    log_hgb_model2.fit(X_hgb_tr2, y_tr2)
    p2 = log_hgb_model2.predict(X_hgb_val2)

    m_mae1 = mean_absolute_error(y_val1, p1)
    m_rmse1 = root_mean_squared_error(y_val1, p1)
    m_mae2 = mean_absolute_error(y_val2, p2)
    m_rmse2 = root_mean_squared_error(y_val2, p2)

    target_exp_results.append({
        "Model": f"Log-Target HGB (lr={lr_val:.2f}, max_iter=200, leaf=15)",
        "September MAE ($)": round(m_mae1, 2),
        "September RMSE ($)": round(m_rmse1, 2),
        "October MAE ($)": round(m_mae2, 2),
        "October RMSE ($)": round(m_rmse2, 2),
        "Mean MAE ($)": round((m_mae1 + m_mae2) / 2, 2),
        "Mean RMSE ($)": round((m_rmse1 + m_rmse2) / 2, 2),
    })

target_df = pd.DataFrame(target_exp_results)
print("=== TARGET TRANSFORMATION EXPERIMENT COMPARISON ===")
display(target_df)


=== TARGET TRANSFORMATION EXPERIMENT COMPARISON ===


,Model,September MAE ($),September RMSE ($),October MAE ($),October RMSE ($),Mean MAE ($),Mean RMSE ($)
0,Linear Ridge (alpha=100) [Benchmark],140.52,622.71,142.06,653.47,141.29,638.09
1,Linear HistGradientBoosting (best) [Benchmark],149.60,628.93,137.81,656.09,143.70,642.51
2,Log-Target Ridge (alpha=10),423.58,898.05,419.95,880.43,421.76,889.24
3,Log-Target Ridge (alpha=30),424.46,899.98,420.58,881.69,422.52,890.84
4,Log-Target Ridge (alpha=100),427.30,905.49,422.85,885.42,425.08,895.46
5,"Log-Target HGB (lr=0.10, max_iter=200, leaf=15)",161.58,633.69,141.09,657.91,151.33,645.80
6,"Log-Target HGB (lr=0.05, max_iter=200, leaf=15)",160.24,633.86,139.14,656.97,149.69,645.41
7,"Log-Target HGB (lr=0.03, max_iter=200, leaf=15)",161.73,634.82,139.48,656.72,150.61,645.77


## 11. EXPERIMENT G: Leakage-Safe Historical Pricing Features

### Objective
Test whether historical lane pricing information explains the extreme rate behavior that the current feature set misses, while enforcing strict temporal leakage isolation.


In [19]:
# 1. Setup Fold Data with Targets
df_f1 = raw_df[pd.to_datetime(raw_df["date"]) < pd.Timestamp("2025-10-01")].copy()
tr1_df = df_f1[pd.to_datetime(df_f1["date"]) < pd.Timestamp("2025-09-01")].copy()
val1_df = df_f1[pd.to_datetime(df_f1["date"]) >= pd.Timestamp("2025-09-01")].copy()

tr2_df = raw_df[pd.to_datetime(raw_df["date"]) < pd.Timestamp("2025-10-01")].copy()
val2_df = raw_df[pd.to_datetime(raw_df["date"]) >= pd.Timestamp("2025-10-01")].copy()

# 2. Compute Leakage-Safe Expanding Features
tr1_hist = compute_historical_pricing_features(tr1_df)
val1_hist = compute_historical_pricing_features(val1_df, history_df=tr1_df)

tr2_hist = compute_historical_pricing_features(tr2_df)
val2_hist = compute_historical_pricing_features(val2_df, history_df=tr2_df)

hist_feature_configs = {
    "A. Current Ridge Champion (Core)": {
        "num": ["distance", "weight"],
        "cat": ["pickup", "delivery", "equipment"],
    },
    "B. Core + Route Historical": {
        "num": [
            "distance",
            "weight",
            "route_prev_count",
            "route_hist_median_rate",
            "route_hist_mean_rate",
            "route_hist_median_rate_per_mile",
        ],
        "cat": ["pickup", "delivery", "equipment"],
    },
    "C. Core + Route + Equip-Route Historical": {
        "num": [
            "distance",
            "weight",
            "route_prev_count",
            "route_hist_median_rate",
            "route_hist_mean_rate",
            "route_hist_median_rate_per_mile",
            "equipment_route_hist_median_rate",
        ],
        "cat": ["pickup", "delivery", "equipment"],
    },
    "D. Core + Route + Equip-Route + Orig/Dest Hist": {
        "num": [
            "distance",
            "weight",
            "route_prev_count",
            "route_hist_median_rate",
            "route_hist_mean_rate",
            "route_hist_median_rate_per_mile",
            "equipment_route_hist_median_rate",
            "origin_hist_median_rate",
            "destination_hist_median_rate",
        ],
        "cat": ["pickup", "delivery", "equipment"],
    },
}

hist_exp_results = []
fitted_hist_models = {}

for name, cfg in hist_feature_configs.items():
    num_cols = cfg["num"]
    cat_cols = cfg["cat"]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]),
                num_cols,
            ),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ]
    )

    # Fold 1
    pipe1 = Pipeline([("preprocessor", preprocessor), ("regressor", Ridge(alpha=100.0, random_state=42))])
    pipe1.fit(tr1_hist[num_cols + cat_cols], y_tr1)
    p1 = pipe1.predict(val1_hist[num_cols + cat_cols])
    mae1 = mean_absolute_error(y_val1, p1)
    rmse1 = root_mean_squared_error(y_val1, p1)

    # Fold 2
    pipe2 = Pipeline([("preprocessor", preprocessor), ("regressor", Ridge(alpha=100.0, random_state=42))])
    pipe2.fit(tr2_hist[num_cols + cat_cols], y_tr2)
    p2 = pipe2.predict(val2_hist[num_cols + cat_cols])
    mae2 = mean_absolute_error(y_val2, p2)
    rmse2 = root_mean_squared_error(y_val2, p2)

    mean_mae = (mae1 + mae2) / 2.0
    mean_rmse = (rmse1 + rmse2) / 2.0

    hist_exp_results.append({
        "Experiment": name,
        "September MAE ($)": round(mae1, 2),
        "September RMSE ($)": round(rmse1, 2),
        "October MAE ($)": round(mae2, 2),
        "October RMSE ($)": round(rmse2, 2),
        "Mean MAE ($)": round(mean_mae, 2),
        "Mean RMSE ($)": round(mean_rmse, 2),
    })
    fitted_hist_models[name] = p2

hist_res_df = pd.DataFrame(hist_exp_results)
print("=== HISTORICAL PRICING FEATURES EXPERIMENT RESULTS ===")
display(hist_res_df)


=== HISTORICAL PRICING FEATURES EXPERIMENT RESULTS ===


,Experiment,September MAE ($),September RMSE ($),October MAE ($),October RMSE ($),Mean MAE ($),Mean RMSE ($)
0,A. Current Ridge Champion (Core),140.52,622.71,142.06,653.47,141.29,638.09
1,B. Core + Route Historical,159.66,623.45,152.80,652.62,156.23,638.04
2,C. Core + Route + Equip-Route Historical,159.65,623.46,152.78,652.63,156.22,638.04
3,D. Core + Route + Equip-Route + Orig/Dest Hist,159.63,623.47,152.82,652.65,156.23,638.06


### Residual Diagnostics on Best Historical Pricing Model (October Holdout)


In [20]:
best_hist_preds_oct = fitted_hist_models["C. Core + Route + Equip-Route Historical"]
val_diag_f2["hist_abs_err"] = np.abs(best_hist_preds_oct - y_val2.values)

m_hist_oct = calc_dist_metrics(y_val2, best_hist_preds_oct, val_diag_f2["hist_abs_err"])

hist_diag_comp = pd.DataFrame([
    {"Model": "Linear Ridge (alpha=100) Core Champion", **{k: round(v, 2) for k, v in m_ridge_oct.items()}},
    {"Model": "Ridge (alpha=100) Core + Historical Pricing", **{k: round(v, 2) for k, v in m_hist_oct.items()}},
])

print("=== RESIDUAL DIAGNOSTICS: CORE CHAMPION VS. HISTORICAL PRICING ===")
display(hist_diag_comp)


=== RESIDUAL DIAGNOSTICS: CORE CHAMPION VS. HISTORICAL PRICING ===


,Model,MAE ($),RMSE ($),Median AE ($),P90 AE ($),P95 AE ($),P99 AE ($),Max AE ($)
0,Linear Ridge (alpha=100) Core Champion,142.06,653.47,60.77,201.91,282.61,1813.70,13884.22
1,Ridge (alpha=100) Core + Historical Pricing,152.78,652.63,72.61,214.78,300.31,1801.87,13819.51


## 12. EXPERIMENT H: Rate-Per-Mile Target Formulation

### Objective
Evaluate a transformed-target formulation where:
$$y_{\text{rpm}} = \frac{\text{posted\_rate}}{\text{distance}}$$
After prediction in rate-per-mile space, predictions are reconstructed:
$$\hat{y} = \hat{y}_{\text{rpm}} \times \text{distance}$$

This is strictly a target transformation (never an input feature). Evaluated on **Core features** across both temporal folds with `Ridge(alpha)` $\in \{30, 100, 300\}$.


In [21]:
core_cols = ["distance", "weight", "pickup", "delivery", "equipment"]

# Rate-per-mile targets
y_tr1_raw = tr1_df["posted_rate"].values
y_val1_raw = val1_df["posted_rate"].values
y_tr2_raw = tr2_df["posted_rate"].values
y_val2_raw = val2_df["posted_rate"].values

y_tr1_rpm = calculate_rate_per_mile_target(y_tr1_raw, tr1_df["distance"].values)
y_tr2_rpm = calculate_rate_per_mile_target(y_tr2_raw, tr2_df["distance"].values)

rpm_exp_results = []
fitted_rpm_models = {}

# 1. Add Champion Baseline
rpm_exp_results.append({
    "Model": "Linear Ridge (alpha=100) [Champion]",
    "Target Form": "posted_rate",
    "alpha": 100.0,
    "September MAE ($)": round(mean_absolute_error(y_val1_raw, best_preds_f1), 2),
    "September RMSE ($)": round(root_mean_squared_error(y_val1_raw, best_preds_f1), 2),
    "October MAE ($)": round(mean_absolute_error(y_val2_raw, best_preds_f2), 2),
    "October RMSE ($)": round(root_mean_squared_error(y_val2_raw, best_preds_f2), 2),
    "Mean MAE ($)": round((mean_absolute_error(y_val1_raw, best_preds_f1) + mean_absolute_error(y_val2_raw, best_preds_f2)) / 2, 2),
    "Mean RMSE ($)": round((root_mean_squared_error(y_val1_raw, best_preds_f1) + root_mean_squared_error(y_val2_raw, best_preds_f2)) / 2, 2),
})

# 2. Evaluate RPM Ridge Candidates
for alpha_val in [30.0, 100.0, 300.0]:
    pipe_rpm1 = Pipeline([("preprocessor", core_preprocessor), ("regressor", Ridge(alpha=alpha_val, random_state=42))])
    pipe_rpm1.fit(tr1_df[core_cols], y_tr1_rpm)
    p_rpm1 = reconstruct_rate_from_rpm(pipe_rpm1.predict(val1_df[core_cols]), val1_df["distance"].values)

    pipe_rpm2 = Pipeline([("preprocessor", core_preprocessor), ("regressor", Ridge(alpha=alpha_val, random_state=42))])
    pipe_rpm2.fit(tr2_df[core_cols], y_tr2_rpm)
    p_rpm2 = reconstruct_rate_from_rpm(pipe_rpm2.predict(val2_df[core_cols]), val2_df["distance"].values)

    m1_mae = mean_absolute_error(y_val1_raw, p_rpm1)
    m1_rmse = root_mean_squared_error(y_val1_raw, p_rpm1)
    m2_mae = mean_absolute_error(y_val2_raw, p_rpm2)
    m2_rmse = root_mean_squared_error(y_val2_raw, p_rpm2)

    rpm_exp_results.append({
        "Model": f"RPM Ridge (alpha={int(alpha_val)})",
        "Target Form": "posted_rate / distance",
        "alpha": alpha_val,
        "September MAE ($)": round(m1_mae, 2),
        "September RMSE ($)": round(m1_rmse, 2),
        "October MAE ($)": round(m2_mae, 2),
        "October RMSE ($)": round(m2_rmse, 2),
        "Mean MAE ($)": round((m1_mae + m2_mae) / 2, 2),
        "Mean RMSE ($)": round((m1_rmse + m2_rmse) / 2, 2),
    })

    fitted_rpm_models[alpha_val] = p_rpm2

rpm_df = pd.DataFrame(rpm_exp_results)
print("=== RATE PER MILE TARGET EXPERIMENT RESULTS ===")
display(rpm_df)


=== RATE PER MILE TARGET EXPERIMENT RESULTS ===


,Model,Target Form,alpha,September MAE ($),September RMSE ($),October MAE ($),October RMSE ($),Mean MAE ($),Mean RMSE ($)
0,Linear Ridge (alpha=100) [Champion],posted_rate,100.0,140.52,622.71,142.06,653.47,141.29,638.09
1,RPM Ridge (alpha=30),posted_rate / distance,30.0,157.06,630.47,159.07,661.69,158.07,646.08
2,RPM Ridge (alpha=100),posted_rate / distance,100.0,155.75,629.90,157.91,661.25,156.83,645.58
3,RPM Ridge (alpha=300),posted_rate / distance,300.0,153.81,629.04,156.05,660.51,154.93,644.77


### Residual Diagnostics on Best Rate-Per-Mile Model (October Holdout)


In [22]:
best_rpm_preds_oct = fitted_rpm_models[100.0]
val_diag_f2["rpm_abs_err"] = np.abs(best_rpm_preds_oct - y_val2_raw)

m_rpm_oct = calc_dist_metrics(y_val2_raw, best_rpm_preds_oct, val_diag_f2["rpm_abs_err"])

rpm_diag_comp = pd.DataFrame([
    {"Model": "Linear Ridge (alpha=100) Core Champion", **{k: round(v, 2) for k, v in m_ridge_oct.items()}},
    {"Model": "RPM Ridge (alpha=100) Core", **{k: round(v, 2) for k, v in m_rpm_oct.items()}},
])

print("=== RESIDUAL DIAGNOSTICS: CORE CHAMPION VS. RATE-PER-MILE TARGET ===")
display(rpm_diag_comp)

# Subgroup diagnostics
top_rate_mask = val_diag_f2["posted_rate"] >= val_diag_f2["posted_rate"].quantile(0.8)
long_dist_mask = val_diag_f2["distance"] >= val_diag_f2["distance"].quantile(0.8)

rpm_subgroup_comp = pd.DataFrame([
    {
        "Subgroup": "Top Rate Quintile (Highest 20%)",
        "Count": top_rate_mask.sum(),
        "Champion MAE ($)": round(val_diag_f2.loc[top_rate_mask, "ridge_abs_err"].mean(), 2),
        "RPM Ridge MAE ($)": round(val_diag_f2.loc[top_rate_mask, "rpm_abs_err"].mean(), 2),
        "Champion RMSE ($)": round(np.sqrt(np.mean(val_diag_f2.loc[top_rate_mask, "ridge_abs_err"]**2)), 2),
        "RPM Ridge RMSE ($)": round(np.sqrt(np.mean(val_diag_f2.loc[top_rate_mask, "rpm_abs_err"]**2)), 2),
    },
    {
        "Subgroup": "Long Distance Quintile (>1650 mi)",
        "Count": long_dist_mask.sum(),
        "Champion MAE ($)": round(val_diag_f2.loc[long_dist_mask, "ridge_abs_err"].mean(), 2),
        "RPM Ridge MAE ($)": round(val_diag_f2.loc[long_dist_mask, "rpm_abs_err"].mean(), 2),
        "Champion RMSE ($)": round(np.sqrt(np.mean(val_diag_f2.loc[long_dist_mask, "ridge_abs_err"]**2)), 2),
        "RPM Ridge RMSE ($)": round(np.sqrt(np.mean(val_diag_f2.loc[long_dist_mask, "rpm_abs_err"]**2)), 2),
    },
])

print("=== SUBGROUP ERROR COMPARISON: RATE-PER-MILE TARGET ===")
display(rpm_subgroup_comp)

# Positivity verification
min_pred_rpm = np.min(best_rpm_preds_oct)
non_pos_count = (best_rpm_preds_oct <= 0).sum()

print("\nPrediction Positivity Verification:")
print(f"  Minimum predicted rate: ${min_pred_rpm:.2f}")
print(f"  Non-positive prediction count: {non_pos_count}")


=== RESIDUAL DIAGNOSTICS: CORE CHAMPION VS. RATE-PER-MILE TARGET ===


,Model,MAE ($),RMSE ($),Median AE ($),P90 AE ($),P95 AE ($),P99 AE ($),Max AE ($)
0,Linear Ridge (alpha=100) Core Champion,142.06,653.47,60.77,201.91,282.61,1813.70,13884.22
1,RPM Ridge (alpha=100) Core,157.91,661.25,65.41,226.74,371.49,1834.76,13940.71


=== SUBGROUP ERROR COMPARISON: RATE-PER-MILE TARGET ===


,Subgroup,Count,Champion MAE ($),RPM Ridge MAE ($),Champion RMSE ($),RPM Ridge RMSE ($)
0,Top Rate Quintile (Highest 20%),971,368.97,427.14,1392.10,1408.11
1,Long Distance Quintile (>1650 mi),971,253.57,314.93,1049.13,1082.71



Prediction Positivity Verification:
  Minimum predicted rate: $166.16
  Non-positive prediction count: 0


## 13. Synthesis & Comprehensive Modeling Recommendations

### 1. Does Rate-Per-Mile Target Formulation Beat $141.29 Mean MAE?
- **No.** The rate-per-mile target formulation worsens cross-fold Mean MAE:
  - **Champion Linear Ridge ($lpha=100.0$)**: **$141.29 Mean MAE** / **$638.09 Mean RMSE**
  - **RPM Ridge ($lpha=300.0$)**: **$154.93 Mean MAE** / **$644.77 Mean RMSE**
  - **RPM Ridge ($lpha=100.0$)**: **$156.83 Mean MAE** / **$645.58 Mean RMSE**
  - **RPM Ridge ($lpha=30.0$)**: **$158.07 Mean MAE** / **$646.08 Mean RMSE**

### 2. Mathematical Explanation for RPM Degradation
- Training on $y / d$ creates an effective loss weighting of $rac{1}{d^2}$ on absolute dollar errors:
  $$\mathcal{L} = \sum_{i} \frac{(y_i - \hat{y}_i)^2}{d_i^2}$$
- This disproportionately penalizes short distances and severely underfits long distances. As a result, MAE on the **Long Distance Quintile** degrades from **$253.57** up to **$314.93 (+24.2%)**.
- All predictions remain strictly positive (minimum rate **$166.16**).

### 3. Definitive Benchmark Ranking Across All Experiments
1. **Overall Champion**: **Standardized Ridge ($lpha=100.0$) on Core Features** (`distance`, `weight`, `pickup`, `delivery`, `equipment`) — **Mean MAE $141.29**, **Mean RMSE $638.09**.
2. **Best Median / Central Regressor**: **Linear HistGradientBoosting** — **Median AE $52.89** (-13% vs Ridge), **October MAE $137.81**.
